In [1]:
import chromadb
chroma_client = chromadb.Client()

In [2]:

from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L12-v2",
    device="cuda",
    normalize_embeddings=True
)

print("SenTransformer Ready")

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 735.53it/s]


SenTransformer Ready


In [3]:
collection = chroma_client.get_or_create_collection(
    name="l12_reranking",
    embedding_function=sentence_transformer_ef
)

In [4]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
print("Data set yüklendi")

Data set yüklendi


In [5]:
doc_texts = []
doc_ids = []

for doc in dataset.docs_iter():
    doc_texts.append(doc.text)
    doc_ids.append(doc.doc_id)

In [6]:
doc_texts

['it was used in landing craft during world war ii and is used today in private boats and training facilities the 6 71 is an inline six cylinder diesel engine the 71 refers to the displacement in cubic inches of each cylinder the firing order of the engine is 1 5 3 6 2 4 the engine s compression ratio is 18 7 1 with a 4 250 inch bore and a 5 00 inch stroke the engine weighs and is 54 inches long 29 inches wide and 41 inches tall at 2 100 revolutions per minute the engine is capable of producing 230 horse power 172 kilowatts v type versions of the 71 series were developed in 1957 the 6 71 is a two stroke engine as the engine will not naturally aspirate air is provided via a roots type blower however on the 6 71t models a turbocharger and a supercharger are utilized fuel is provided by unit injectors one per cylinder the amount of fuel injected into the engine is controlled by the engine s governor the engine cooling is via liquid in a water jacket in a boat cool external water is pumped

### Vector Database Storage

In [7]:
from tqdm import tqdm

BATCH_SIZE = 5000
total_doc = len(doc_texts)

for i in tqdm(range(0, total_doc, BATCH_SIZE)):
    batch_texts = doc_texts[i: i + BATCH_SIZE]
    batch_ids = doc_ids[i: i + BATCH_SIZE]

    collection.add(
        documents=batch_texts,
        ids=batch_ids
    )

print("Vectorization!")

100%|██████████| 74/74 [1:12:09<00:00, 58.51s/it] 

Vectorization!


In [8]:
queries = []

for query in dataset.queries_iter():
    queries.append(query.text)

In [11]:
results_15 = collection.query(
    query_texts=queries,
    n_results=15
)

In [15]:
import gc
del results_20
gc.collect()

0

In [13]:
from collections import defaultdict
query_id_vs_text = defaultdict(str)

for query in dataset.queries_iter():
    query_id_vs_text[query.query_id] = query.text

query_id_vs_text = dict(query_id_vs_text)

In [16]:
doc_id_vs_text = defaultdict(str)

for doc in dataset.docs_iter():
    doc_id_vs_text[doc.doc_id] = doc.text

In [17]:
results_15['ids']

[['516871',
  '647478',
  '2129172',
  '806300',
  '2357011',
  '1222510',
  '806263',
  '2433706',
  '708880',
  '775424',
  '434612',
  '2320659',
  '2356422',
  '1987805',
  '1423498'],
 ['188629',
  '251586',
  '2397950',
  '2061524',
  '272146',
  '1948304',
  '2437665',
  '1278316',
  '1614942',
  '1363774',
  '1399078',
  '1379412',
  '492051',
  '1936742',
  '1152359'],
 ['13898',
  '785960',
  '1382760',
  '941146',
  '802532',
  '1048798',
  '1989378',
  '1874948',
  '2031249',
  '462902',
  '1263356',
  '1334004',
  '166703',
  '1263194',
  '259523'],
 ['316959',
  '562691',
  '1326263',
  '2156815',
  '687712',
  '42143',
  '1639962',
  '1469362',
  '729083',
  '1422464',
  '1921875',
  '1999122',
  '714978',
  '1978987',
  '2224682'],
 ['515031',
  '706326',
  '780987',
  '832086',
  '842375',
  '860147',
  '828931',
  '833668',
  '754830',
  '823332',
  '843406',
  '837562',
  '1662828',
  '836749',
  '888195'],
 ['2267777',
  '421974',
  '1630942',
  '1209576',
  '232292

In [20]:
query_ids = [query.query_id for query in dataset.queries_iter()]

In [22]:
query_doc_pairs = defaultdict(list)

for idx in range(len(query_ids)):
    for doc_id in results_15['ids'][idx]:
        query_id = query_ids[idx]
        query_doc_pairs[query_id].append((query_id_vs_text[query_id], doc_id_vs_text[doc_id]))

query_doc_pairs = dict(query_doc_pairs)

### CrossEncoder

In [25]:
from sentence_transformers import CrossEncoder
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1966.98it/s]


In [26]:
query_score_dict = defaultdict(list)

for query_id in query_ids:
    query_score_dict[query_id] = list(model.predict(query_doc_pairs[query_id]))

In [27]:
query_score_dict["123839"]

[np.float32(-0.45067456),
 np.float32(-0.17859712),
 np.float32(-0.33984625),
 np.float32(6.4627924),
 np.float32(0.458524),
 np.float32(2.2397873),
 np.float32(4.384878),
 np.float32(1.5397115),
 np.float32(-2.3310945),
 np.float32(-1.3943533),
 np.float32(-0.76778865),
 np.float32(1.908722),
 np.float32(0.89662874),
 np.float32(-9.491181),
 np.float32(-9.244002)]

In [31]:
retrieved_docs_with_scores = defaultdict(list)

for query_idx in range(len(query_ids)):
    tuple_list = []
    query_id = query_ids[query_idx]

    for doc_idx in range(15):
        tuple_list.append((query_score_dict[query_id][doc_idx], results_15['ids'][query_idx][doc_idx]))

    retrieved_docs_with_scores[query_id] = sorted(tuple_list, reverse=True)

retrieved_docs_with_scores = dict(retrieved_docs_with_scores)

In [32]:
retrieved_docs_with_scores

{'123839': [(np.float32(6.4627924), '806300'),
  (np.float32(4.384878), '806263'),
  (np.float32(2.2397873), '1222510'),
  (np.float32(1.908722), '2320659'),
  (np.float32(1.5397115), '2433706'),
  (np.float32(0.89662874), '2356422'),
  (np.float32(0.458524), '2357011'),
  (np.float32(-0.17859712), '647478'),
  (np.float32(-0.33984625), '2129172'),
  (np.float32(-0.45067456), '516871'),
  (np.float32(-0.76778865), '434612'),
  (np.float32(-1.3943533), '775424'),
  (np.float32(-2.3310945), '708880'),
  (np.float32(-9.244002), '1423498'),
  (np.float32(-9.491181), '1987805')],
 '188629': [(np.float32(8.445473), '188629'),
  (np.float32(3.8551636), '2437665'),
  (np.float32(0.7997365), '251586'),
  (np.float32(-0.66893935), '2061524'),
  (np.float32(-0.7973872), '1948304'),
  (np.float32(-0.8317001), '1379412'),
  (np.float32(-2.1443884), '1936742'),
  (np.float32(-2.170946), '1399078'),
  (np.float32(-2.263849), '492051'),
  (np.float32(-2.370374), '2397950'),
  (np.float32(-2.9040346), 

In [33]:
most_related_10_with_scores = defaultdict(list)

for query_id in query_ids:
    most_related_10_with_scores[query_id] = retrieved_docs_with_scores[query_id][:10]

most_related_10_with_scores = dict(most_related_10_with_scores)

In [35]:
most_related_5_with_scores = defaultdict(list)

for query_id in query_ids:
    most_related_5_with_scores[query_id] = retrieved_docs_with_scores[query_id][:5]

most_related_5_with_scores = dict(most_related_5_with_scores)

In [37]:
qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

query_ids = [query.query_id for query in dataset.queries_iter()]

class Scoredoc:
    def __init__(self, doc_id, score):
        self.doc_id = doc_id
        self.score = score

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

In [38]:
del dataset
del queries
del doc_texts
del doc_ids
del model
gc.collect()

2006

In [39]:
from collections import defaultdict
from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

def recall(found_list: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for mytuple in found_list:
        if mytuple[1] in test_set:
            counter += 1

    return (counter / len(test_list)) * 100

def precision(found_list: list, test_list: list) -> float:
    if len(found_list) == 0:
        return 0

    counter = 0

    test_set = set(test_list)

    for mytuple in found_list:
        if mytuple[1] in test_set:
            counter += 1

    return (counter / len(found_list)) * 100

def precision_AP(found_docs: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for mytuple in found_docs:
        if mytuple[1] in test_set:
            counter += 1

    return counter / len(found_docs)

def AP(found_docs: list, test_list: list, value: int) -> float:
    total = 0

    for i in range(1, value + 1):
        if i < len(found_docs):
            precision_k = precision_AP(found_docs[:i], test_list)
            total += precision_k * (found_docs[i - 1][1] in set(test_list))

    return total / len(test_list)

def get_ndcg_list(most_dict: dict, value: int, query_ids: list, score_doc_dict: dict) -> list:
    doc_id_score_dict = defaultdict(float)
    ndcg_list = []

    for query_id in query_ids:
        scoredoc_object_list = score_doc_dict[query_id]

        for scoreddoc_object in scoredoc_object_list:
            doc_id_score_dict[scoreddoc_object.doc_id] = scoreddoc_object.score

        model_score_tuple_list = most_dict[query_id]
        y_score, y_true = [], []

        for mytuple in model_score_tuple_list:
            doc_id = mytuple[1]
            score = mytuple[0]

            y_score.append(score)
            y_true.append(doc_id_score_dict[doc_id])

        if len(y_score) == 0:
            ndcg_list.append(0.0)
            continue

        if len(y_score) == 1:
            y_true.append(0.0)
            y_score.append(0.0)

        if len(y_true) == len(y_score):
            ndcg_list.append(ndcg_score(np.asarray([y_true]), np.asarray([y_score]), k=value))
        else:
            print("There is a problem with query", query_id)

    return ndcg_list

- for top 10

In [40]:
recall_10_list = []

for query_id in query_ids:
    recall_10_list.append(recall(most_related_10_with_scores[query_id], qrels_dict[query_id]))

In [46]:
precision_10_list = []

for query_id in query_ids:
    precision_10_list.append(precision(most_related_10_with_scores[query_id], qrels_dict[query_id]))

In [48]:
AP_10_list = []

for query_id in query_ids:
    AP_10_list.append(AP(most_related_10_with_scores[query_id], qrels_dict[query_id], 10))

In [50]:
NDCG_10_list = get_ndcg_list(most_related_10_with_scores, 10, query_ids, score_doc_dict)

In [53]:
recall_5_list = []

for query_id in query_ids:
    recall_5_list.append(recall(most_related_5_with_scores[query_id], qrels_dict[query_id]))

precision_5_list = []

for query_id in query_ids:
    precision_5_list.append(precision(most_related_5_with_scores[query_id], qrels_dict[query_id]))

AP_5_list = []

for query_id in query_ids:
    AP_5_list.append(AP(most_related_5_with_scores[query_id], qrels_dict[query_id], 5))

NDCG_5_list = get_ndcg_list(most_related_5_with_scores, 5, query_ids, score_doc_dict)

In [54]:
def base_df(
        query_ids: list,
        recall_5: list,
        precision_5: list,
        AP_5: list,
        NDCG_5: list,
        recall_10: list,
        precision_10: list,
        AP_10: list,
        NDCG_10: list
) -> pd.DataFrame:

    df = pd.DataFrame({
        "Query_ID": query_ids,
        "recall_5": recall_5,
        "precision_5": precision_5,
        "AP_5": AP_5,
        "NDCG_5": NDCG_5,
        "recall_10": recall_10,
        "precision_10": precision_10,
        "AP_10": AP_10,
        "NDCG_10": NDCG_10
    })

    df["f_score_5"] = 2 * df["recall_5"] * df["precision_5"] / (df["recall_5"] + df["precision_5"])
    df["f_score_10"] = 2 * df["recall_10"] * df["precision_10"] / (df["recall_10"] + df["precision_10"])

    df["f_score_5"] = df["f_score_5"].fillna(0)
    df["f_score_10"] = df["f_score_10"].fillna(0)

    return df

In [55]:
df = base_df(
    query_ids=query_ids,
    recall_5=recall_5_list,
    precision_5=precision_5_list,
    AP_5=AP_5_list,
    NDCG_5=NDCG_5_list,
    recall_10=recall_10_list,
    precision_10=precision_10_list,
    AP_10=AP_10_list,
    NDCG_10=NDCG_10_list
)

df

,Query_ID,recall_5,precision_5,AP_5,NDCG_5,recall_10,precision_10,AP_10,NDCG_10,f_score_5,f_score_10
0,123839,33.333333,40.0,0.333333,1.000000,33.333333,20.0,0.333333,1.000000,36.363636,25.000000
1,188629,16.666667,20.0,0.166667,1.000000,16.666667,10.0,0.166667,0.941792,18.181818,12.500000
2,13898,33.333333,40.0,0.333333,0.000000,33.333333,20.0,0.333333,0.000000,36.363636,25.000000
3,316959,22.222222,40.0,0.222222,0.997278,22.222222,20.0,0.222222,0.997278,28.571429,21.052632
4,515031,7.142857,20.0,0.071429,0.688473,7.142857,10.0,0.071429,0.689304,10.526316,8.333333
...,...,...,...,...,...,...,...,...,...,...,...
1439,896124,12.500000,20.0,0.125000,0.957719,12.500000,10.0,0.125000,0.968451,15.384615,11.111111
1440,12319,4.545455,20.0,0.045455,0.886219,4.545455,10.0,0.045455,0.800051,7.407407,6.250000
1441,4421,0.000000,0.0,0.000000,0.897969,0.000000,0.0,0.000000,0.676304,0.000000,0.000000
1442,296526,10.000000,20.0,0.033333,0.903982,20.000000,20.0,0.055556,0.886751,13.333333,20.000000


In [56]:
df.isna().sum()

Query_ID        0
recall_5        0
precision_5     0
AP_5            0
NDCG_5          0
recall_10       0
precision_10    0
AP_10           0
NDCG_10         0
f_score_5       0
f_score_10      0
dtype: int64

In [57]:
def create_parquet_df(method_name: str, df: pd.DataFrame) -> pd.DataFrame:
    mydict = {
        "Method": method_name,
        "recall_5_mean": df["recall_5"].mean(),
        "recall_5_std": df["recall_5"].std(),
        "recall_5_max": df["recall_5"].max(),
        "recall_5_min": df["recall_5"].min(),
        "recall_10_mean": df["recall_10"].mean(),
        "recall_10_std": df["recall_10"].std(),
        "recall_10_max": df["recall_10"].max(),
        "recall_10_min": df["recall_10"].min(),
        "precision_5_mean": df["precision_5"].mean(),
        "precision_5_std": df["precision_5"].std(),
        "precision_5_max": df["precision_5"].max(),
        "precision_5_min": df["precision_5"].min(),
        "precision_10_mean": df["precision_10"].mean(),
        "precision_10_std": df["precision_10"].std(),
        "precision_10_max": df["precision_10"].max(),
        "precision_10_min": df["precision_10"].min(),
        "f_score_5_mean": df["f_score_5"].mean(),
        "f_score_5_std": df["f_score_5"].std(),
        "f_score_5_max": df["f_score_5"].max(),
        "f_score_5_min": df["f_score_5"].min(),
        "f_score_10_mean": df["f_score_10"].mean(),
        "f_score_10_std": df["f_score_10"].std(),
        "f_score_10_max": df["f_score_10"].max(),
        "f_score_10_min": df["f_score_10"].min(),
        "MAP_5": df["AP_5"].mean(),
        "MAP_10": df["AP_10"].mean(),
        "NDCG_5_mean": df["NDCG_5"].mean(),
        "NDCG_5_std": df["NDCG_5"].std(),
        "NDCG_5_max": df["NDCG_5"].max(),
        "NDCG_5_min": df["NDCG_5"].min(),
        "NDCG_10_mean": df["NDCG_10"].mean(),
        "NDCG_10_std": df["NDCG_10"].std(),
        "NDCG_10_max": df["NDCG_10"].max(),
        "NDCG_10_min": df["NDCG_10"].min()
    }

    df_parquet = pd.DataFrame(mydict, index=[0])

    return df_parquet

In [58]:
df_parquet = create_parquet_df("minilm_l12_reranking", df)

In [59]:
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,minilm_l12_reranking,14.16927,12.872118,66.666667,0.0,17.231414,15.961054,100.0,0.0,31.260388,...,0.11453,0.134332,0.750344,0.352292,1.0,0.0,0.752008,0.307466,1.0,0.0


In [60]:
df_parquet.to_parquet("minilm_l12_reranking.parquet")